In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=64, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(in_channels=64, out_channels=32, kernel_size=5, padding=2)
        self.conv3 = nn.Conv2d(in_channels=64, out_channels=32, kernel_size=5, padding=2)

        self.conv4 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, stride=2, padding=1)
        self.conv5 = nn.Conv2d(in_channels=128, out_channels=256, kernel_size=3, stride=2, padding=1)
        self.conv6 = nn.Conv2d(in_channels=256, out_channels=512, kernel_size=3, stride=1, padding=1)
        self.conv7 = nn.AvgPool2d(kernel_size=2)
        self.fc = nn.Linear(512 * 4 * 4, 10)  # Adjusted for final feature map size

    def forward(self, x):
        # First convolution with ReLU
        x1 = F.relu(self.conv1(x))
        
        # Second convolution 5x5
        x2 = F.relu(self.conv2(x1))
        
        # Third convolution 5x5
        x3 = F.relu(self.conv3(x1))
        
        # Concatenate along the channel dimension
        x_cat = torch.cat([x2, x3], dim=1)
        
        # Final convolution with stride 2
        x4 = F.relu(self.conv4(x_cat))
        
        x5 = F.relu(self.conv5(x4))

        x6 = F.relu(self.conv6(x5))

        x7 = self.conv7(x6)

        x7 = x7.view(x7.size(0), -1)  # Flatten to (batch_size, 512*4*4)
        out = self.fc(x7)

        
        return out

# Initialize the network
net = Net()



In [ ]:
# Example input: batch size 32, 3 channels (RGB), height 32, width 32
input_tensor = torch.randn(32, 3, 32, 32)

# Forward pass
output = net(input_tensor)

# Print output shape
print(output.shape)


In [ ]:
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder

transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])
data_dir = '/kaggle/input/cifar10/cifar10'

# Tạo DataLoader cho tập train và test
trainset = ImageFolder(root=f"{data_dir}/train", transform=transform)
trainloader = DataLoader(trainset, batch_size=32, shuffle=True, num_workers=2)

testset = ImageFolder(root=f"{data_dir}/test", transform=transform)
testloader = DataLoader(testset, batch_size=32, shuffle=False, num_workers=2)



In [ ]:
# 5. Huấn luyện mạng
def train(net, trainloader, criterion, optimizer, device, num_epochs=10):
    net.train()
    for epoch in range(num_epochs):
        running_loss = 0.0
        for i, data in enumerate(trainloader, 0):
            inputs, labels = data
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = net(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            if i % 100 == 99:
                print(f"[{epoch + 1}, {i + 1}] loss: {running_loss / 100:.3f}")
                running_loss = 0.0
    print('Finished Training')


# 6. Đánh giá mạng trên tập test
def evaluate(net, testloader, device):
    net.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for data in testloader:
            images, labels = data
            images, labels = images.to(device), labels.to(device)
            outputs = net(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    print(f'Accuracy of the network on the 10000 test images: {100 * correct / total:.2f}%')


In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
net = Net().to(device)




In [ ]:
# 4. Xác định hàm mất mát và optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters(), lr=0.001)
train(net, trainloader, criterion, optimizer, device, num_epochs=10)
evaluate(net, testloader, device)
